In [ ]:
side = "Left"
version = "0.1"

import pandas as pd
import re
from pprint import pprint

In [ ]:
#This script first trains an NLP model for peripheral left abnormal using labeled US text reports
#It then use the model to find US reports that are positive.


#data/peripheral_left has all labeled US text report by BCH
from pathlib import Path

def read_us_split(split_dir):
    split_dir = Path(split_dir)
    texts = []
    labels = []
    for label_dir in ["pos", "neg"]:
        for text_file in (split_dir/label_dir).iterdir():
            texts.append(text_file.read_text())
            labels.append(0 if label_dir == "neg" else 1)
    return texts, labels

#train_texts, train_labels = read_us_split('data/peripheral_left') #all labeled by BCH

def pull_reports():
    print(f'Loading 2011_search_US-Renal.xlsx')
    df = pd.read_excel('2011_search_US-Renal.xlsx')
    for year in range(2012,2025):
        filename = f'{year}_search_US-Renal.xlsx'

        print(f'Loading {filename}')
        temp = pd.read_excel(filename)

        df = pd.concat([df,temp])

    return df

accession_df = pull_reports()

label_df = pd.read_excel('matched_accessions.xlsx')
label_df.set_index('Accession',inplace=True)
label_dict = label_df.to_dict('index')




In [ ]:
def extract_sections(report):

    # Create a dictionary to store the extracted sections
    extracted_data = {}

    # Define a regular expression pattern to capture each section
    #pattern = re.compile(r'(?P<section>[A-Z][A-Z\s]*):\s*(?P<content>.*?)(?=(?:[A-Z][A-Z\s]*):|\Z)', re.DOTALL)
    pattern = re.compile(r'(?P<section>[A-Za-z\s]+):\s*(?P<content>.*?)(?=(?:[A-Za-z\s]+):|\Z)', re.DOTALL)


    # Find all matches in the report
    matches = pattern.finditer(report)

    for match in matches:
        section = match.group('section').strip()
        content = match.group('content').strip()
        extracted_data[section.upper()] = content.replace("\n","").replace("_x000D_","")


    if len(extracted_data) == 0: print(report)

    return extracted_data

def extract_grade(grade):
    try:
        grade = float(grade.split(" ")[1])
        return grade
    
    except:
        return 0


def get_labeled_data(label_dict, accession_df, side):
    texts = []
    labels = []

    include_sections = ['FINDINGS', 'IMPRESSION', 'OBSERVATIONS', 'UTD SCORE', 
                        'RIGHT KIDNEY', 'LEFT KIDNEY', 'BLADDER', 'P REFLUX REPAIR', 
                        'INDICATIONS', 'BIOMETRY', 'RIGHT KIDNEY AND URETER', 'CALYCEAL DILATION', 
                        'MAJOR CALYCES', 'MINOR CALYCES', 'PARENCHYMAL THICKNESS', 'PARENCHYMAL APPEARANCE', 
                        'URETERS', 'LEFT KIDNEY AND URETER', 'UTD CLASSIFICATION RIGHT', 'UTD CLASSIFICATION LEFT', 
                        'DOI', 'HTTP', 'RENAL ULTRASOUND', 'EXAMINATION', 'UTD', 'OBSERVATION', 
                        'RECOMMENDATION', 'RENAL SONOGRAM', 'US REASON', 'DATE', 'GENDER', 'RIGHT', 'LEFT', 'PELVIS AND URINARY BLADDER',
                        'ADDENDUM', 'THE UNCHANGED URETEROCELE WAS ADDED TO THE IMPRESSION SECTION OF THIS REPORT AS FOLLOWS', 'STUDY', 
                        'KIDNEYS AND URETERS', 'OTHER', 'IN PRESSION', 'URINARY BLADDER', 'STATUS POST PRIOR PARTIAL RIGHT NEPHRECTOMY', 
                        'P', 'AND UTD CLASSIFICATION LEFT', 'LEFT UPPER ABDOMEN', 'NOTE', 'THE REPORT CORRECTED AND ADDENDED AS FOLLOWS', 
                        'WITH THE FOLLOWING OVERALL CYST DIMENSIONS', 'WITH THE FOLLOWING APPROXIMATE DIMENSIONS', 'INFORMATION OBTAINED FROM THE EMR', 
                        'KIDNEYS', 'ADDITIONAL HISTORY AS PER THE EMR', 'WITHIN THIS LIMITATION', 'UTD CLASSIFICATION', 'ADDITIONAL HISTORY',
                        'LEFT RENAL AP DIAMETER', 'HYDRONEPHROSIS ADDITIONAL INFORMATION OBTAINED PER THE EMR', 'RIGHT KIDNEY LENGTH', 
                        'LEFT RENAL LENGTH', 'IN THE MID', 'S NAME', 'SEVERAL CHANGES WERE APPARENT', 'DIAGNOSIS', 'LEFT RENAL FOSSA', 
                        'UPPER MOIETY', 'LOWER MOIETY', 'S NAME AND MEDICAL RECORD NUMBER', 'PRENATAL HYDRONEPHROSIS MOMS MRN', 
                        'AP RENAL PELVIS DIAMETER', 'SEVERE LEFT HYDRONEPHROSIS WITH PARENCHYMAL THINNING', 'MRN', 'U', 'SPINE US', 
                        'STUDY DATE', 'COMPARISONS', 
                        'THE URINARY BLADDER IS SIMILARLY TRABECULATED IN KEEPING WITH HISTORY OF POSTERIOR URETHRAL VALVES IMPRESSION', 
                        'EXAM', 'S NAME AND MR NUMBER', 'INDICATION PER ELECTRONIC REQUISITION', 'HEAD ULTRASOUND', 'GRADING IS BASED ON', 
                        'RENAL ULTRASONOGRAPHY', 'TRANSPLANT KIDNEY', 'ABDOMINAL AORTA', 'INFERIOR VENA CAVA', 'ARTERIES', 'TRANSPLANT RENAL VEIN', 
                        'PER EMR', 'RT KIDNEY', 'LT KIDNEY', 'RENAL LENGTH', 'IMPRESSIONS', 'AND HER MATERNAL MRN', 'OV ADDITIONAL INFORMATION OBTAINED FROM THE EMR', 
                        'RENAL GROWTH', 'UTD REFERENCE', 'ADDITIONAL INFORMATION OBTAINED PER THE EMR', 'SPINE', 'MOTHER', 'NEW DX', 
                        'UTD CLASSIFICATION CORRECTION', 'INDICATION AS PER REQUISITION', 'HARRISON', 'LEFT KIDNEY MEASUREMENT', 'MATERNAL INFORMATION', 
                        'MPRESSION', 'ADDENDUM TO CORRECT LATERALITY ERROR IN IMPRESSION', 'DX', 'SPINE ULTRASOUND', 'S MRN', 'UTD CLASSIFICATION RIGHT KIDNEY', 
                        'UTD CLASSIFICATION LEFT KIDNEY', 'MATERNAL SONOGRAM PERFORMED IN THE AFCC', 'RIGHT RENAL MOIETY', 'LEFT RENAL MOIETY', 'FREE FLUID', 
                        'S MEDICAL RECORD NUMBER', 'PER EMR REVIEW', 'PRELIMINARY REPORT', 
                        'CM AS WELL AS MILD CENTRAL AND PERIPHERAL CALYCEAL DILATATION WITH UTD CLASSIFICATION AS BELOW', 'RENAL STUDY', 
                        'IMPRESSION SHOULD READ AS FOLLOWS', 'ADDITIONAL HISTORY PER EHR', 'PELVIS DILATION', 'URETER', 'WITH THE FOLLOWING FINDINGS', 
                        'ADDITIONAL HISTORY ON THE REQUISITION IS AS FOLLOWS', 'RENAL DOPPLER', 'RENAL AND BLADDER ULTRASOUND', 'CYCLIC VCUG', 
                        'FLUOROSCOPY', 'TIME', 'RADIATION EXPOSURE', 'COLOR AND SPECTRAL DOPPLER ULTRASOUND OF THE KIDNEYS DEMONSTRATES THE FOLLOWING', 
                        'S MEDICAL RECORD', 'UTD CLASSIFICATION REFERENCE', 'BILATERAL HIP ULTRASOUND', 'TECHNICAL NOTE', 'IMAGING', 'YANFEI MRN', 
                        'FASHION', 'PRIOR BCH PROCEDURES INCLUDE THE FOLLOWING', 'FETAL FINDINGS', 'MATERNAL MRN', 'KAITLYN MRN', 
                        'ADDITIONAL CLINICAL HISTORY OBTAINED FROM EMR', 'ADDITIONAL HISTORY PER EMR', 'HIPS', 'RIGHT COLLECTING SYSTEM', 
                        'LEFT COLLECTING SYSTEM', 'TIME FRAME', 'TESTING', 'OV FETAL HYDRONEPHROSIS', 'KURTZ MOM', 'LEFT KIDNEY LENGTH', 
                        'THE FOLLOWING OBSERVATIONS ARE MADE', 'UPPER POLE', 'LOWER POLE', 'OV WITH RUS RE', 'RIGHT KIDNEY UTD SCORE', 
                        'LEFT KIDNEY UTD SCORE', 'PRIOR PROCEDURES AT BCH', 'THE FOLLOWING FINDINGS ARE NOTED', 'LEFT UTD CLASSIFICATION', 
                        'NORMAL APPEARING RIGHT KIDNEY RIGHT UTD CLASSIFICATION']
    all_keys = []
    for accession in label_dict:

        #report
        filtered_df = accession_df[accession_df['Accession Number'].astype(str) == str(accession)]
        report = filtered_df['Report Text'].iloc[0]
        report_dict = extract_sections(report)

        report_filtered = ""
        for key in report_dict:
            if key in include_sections: report_filtered += f'{key}: {report_dict[key]} '

        texts.append(report_filtered)

        #label
        grade = extract_grade(label_dict[accession][side])
        labels.append(1 if grade >= 2 else 0)

    return labels, texts

labels, texts = get_labeled_data(label_dict, accession_df, side)


In [ ]:
#We first used the script to find all the P2/P3 candidates: find_p2p3_candidates
#then read reports that are candidate for p2/p3. We will use the trained NLP model to identify those MRN that really P2
#Note that the report name is MRN.txt, so we can easily retrieve MRN for each report
import os
def read_us_p2p3(p2p3_dir):
    p2p3_directory = Path(p2p3_dir)
    texts = []
    labels = []
    mrns = []
    count = 0
    for text_file in p2p3_directory.iterdir():
        report_file_name = os.path.basename(text_file)
        mrn, file_extension = os.path.splitext(report_file_name)
        texts.append(text_file.read_text())
        labels.append(1) #do not really the label
        mrns.append(mrn)
        count = count+1
    return texts, labels, mrns

p2p3_texts, p2p3_labels, mrns = read_us_p2p3('p2_p3_reports_candidates_from_all_reports') 


In [ ]:

from sklearn.model_selection import train_test_split
train_texts, val_texts, train_labels, val_labels = train_test_split(texts, labels, test_size=.3, random_state=5) #last parameter is seed
test_texts, val_texts, test_labels, val_labels = train_test_split(val_texts, val_labels, test_size=.5, random_state=5)

In [ ]:
#NLP needs a tokenizer from pretrained BERT
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.2", model_max_length=512)
train_encodings = tokenizer(train_texts, truncation=True, padding=True)
val_encodings = tokenizer(val_texts, truncation=True, padding=True)
p2p3_encodings = tokenizer(p2p3_texts, truncation=True, padding=True)

In [ ]:
#Convert the datasets to tensors that can be used by PyTorch
import torch

class us1500Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = us1500Dataset(train_encodings, train_labels)
val_dataset = us1500Dataset(val_encodings, val_labels)

p2p3_dataset = us1500Dataset(p2p3_encodings, p2p3_labels)

In [ ]:

#An example of  data after tokenization
p2p3_dataset[12]


In [ ]:
#validation accuracy function. This function is used to calculate eval accuracy at each epoch
import numpy as np
from datasets import load_metric

metric = load_metric("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
from transformers import BertConfig
my_config = BertConfig.from_pretrained("dmis-lab/biobert-base-cased-v1.2")
my_config

In [ ]:
my_config.hidden_dropout_prob=0.2

In [ ]:
torch.manual_seed(0)

from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',          # output directory
    num_train_epochs=11,#11,              # total number of training epochs
    per_device_train_batch_size=8,  # batch size per device during training
    per_device_eval_batch_size=8,   # batch size for evaluation
    warmup_steps=500,                # number of warmup steps for learning rate scheduler
    weight_decay=0.01,               # strength of weight decay
    logging_dir='./logs',            # directory for storing logs
    logging_steps=10,
    evaluation_strategy="epoch",
)


model = AutoModelForSequenceClassification.from_pretrained("dmis-lab/biobert-base-cased-v1.2", config=my_config)

trainer = Trainer(
    model=model,                         # the instantiated 🤗 Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=train_dataset,         # training dataset
    eval_dataset=val_dataset,             # evaluation dataset
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
p2p3_preds = trainer.predict(p2p3_dataset)

y_preds  = []
positive_count = 0
true_p2_mrns = []
for index, label in enumerate(p2p3_preds.label_ids):
    prediction = np.argmax(p2p3_preds.predictions[index], axis=-1)
    y_preds.append(prediction)
    if prediction == 1:
        positive_count = positive_count + 1
        print(mrns[index])
        true_p2_mrns.append(mrns[index])
print(positive_count)

In [ ]:
import pandas as pd

new_mrn_list_df = pd.DataFrame({"MRN": true_p2_mrns})
new_mrn_list_df.to_csv("new_mrns.csv")